### Introduction to Post-Training and RLHF Overview


1. RL is basicall

### Post Training Basic Example

In [ ]:
# ! pip install transformers
# ! pip install datasets
# ! pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# ! pip install accelerate peft trl bitsandbytes deepspeed

In [ ]:
import torch; print('PyTorch Version:', torch.__version__); print('CUDA Available:', torch.cuda.is_available())

PyTorch Version: 2.5.1+cu121
CUDA Available: True


In [ ]:
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

/home/hice1/vchopra37/scratch/envs/learning_python_env/lib/python3.12/site-packages/huggingface_hub/constants.py:299: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


In [ ]:
ASSETS_DIR = Path('./offline_assets').resolve()
model_name = 'gpt2'
model_name

'gpt2'

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

### Step-1- Testing the tokenizer

Let us test the encoding and decoding process

In [ ]:
tokenizer.pad_token

In [ ]:
text = "Hello, this is the first step of RLHF training."
tokens = tokenizer(text)
print(f"Encoded tokens: {tokens}")
print(f"Decoded Version: {tokenizer.decode(tokens)}")

Encoded tokens: {'input_ids': [15496, 11, 428, 318, 262, 717, 2239, 286, 45715, 29567, 3047, 13], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
Decoded Version: Hello, this is the first step of RLHF training.


In [ ]:
tokenizer.pad_token 

texts = [
    "Hello",
    "Hello, this is the first step of RLHF training."
]

tokens = tokenizer(
    texts,
    padding=True,
    return_tensors="pt"
)
print(tokens, len(tokens))

{'input_ids': tensor([[15496, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
         50256, 50256],
        [15496,    11,   428,   318,   262,   717,  2239,   286, 45715, 29567,
          3047,    13]]), 'attention_mask': tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])} 2


In [ ]:
print(tokens["input_ids"][0])
print(tokens["attention_mask"][0])
print(tokenizer.decode(tokens['input_ids'][0]))

tensor([15496, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256])
tensor([1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
Hello<|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|><|endoftext|>


### Step-2:- Loading the Dataset

In [ ]:
from datasets import load_dataset

In [ ]:
ds = load_dataset('stanfordnlp/sst2')
ds

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [ ]:
ds_train, ds_val, ds_test = ds['train'], ds['validation'], ds['test']
print(ds_train.shape, ds_val.shape, ds_test.shape)

(67349, 3) (872, 3) (1821, 3)


In [ ]:
SAMPLE = 6
ds_train[SAMPLE]

{'idx': 6,
 'sentence': 'demonstrates that the director of such hollywood blockbusters as patriot games can still turn out a small , personal film with an emotional wallop . ',
 'label': 1}

### Step-3: - Preparing the Dataset

#### Step-3.1:- Tokenizing the dataset

In [ ]:
def tokenize(batch):
    return tokenizer(batch['sentence'])

map_kwargs = {
    'batched' :True,
    'batch_size' :512,
    'remove_columns': ['idx', 'sentence', 'label']
    
}

tokenized_dataset_train = ds_train.map(tokenize, **map_kwargs)
tokenized_dataset_val = ds_val.map(tokenize, **map_kwargs)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

In [ ]:
tokenized_dataset_train[0]

{'input_ids': [24717, 649, 3200, 507, 422, 262, 21694, 4991, 220],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

#### Step-3.2: - Decoding from the dataset

In [ ]:
for i, seq in enumerate(tokenized_dataset_train[5:10]['input_ids']):
    print(f'{i+1}: {tokenizer.decode(seq)}')

1: that 's far too tragic to merit such superficial treatment 
2: demonstrates that the director of such hollywood blockbusters as patriot games can still turn out a small , personal film with an emotional wallop . 
3: of saucy 
4: a depressed fifteen-year-old 's suicidal poetry 
5: are more deeply thought through than in most ` right-thinking ' films 


#### Step-3.3 :-  Filter out tweets shorter than 5 tokens

In [ ]:
print(len(tokenized_dataset_train), len(tokenized_dataset_val))
tokenized_dataset_train = tokenized_dataset_train.filter(lambda x: len(x['input_ids']) > 5)
tokenized_dataset_val = tokenized_dataset_val.filter(lambda x: len(x['input_ids']) > 5)

67349 872


Filter:   0%|          | 0/67349 [00:00<?, ? examples/s]

Filter:   0%|          | 0/872 [00:00<?, ? examples/s]

In [ ]:
print(len(tokenized_dataset_train), len(tokenized_dataset_val))

49401 867


In [ ]:
tokenized_dataset_train.set_format(type='torch')
tokenized_dataset_val.set_format(type='torch')
tokenized_dataset_train[0], tokenized_dataset_train[5:10]

({'input_ids': tensor([24717,   649,  3200,   507,   422,   262, 21694,  4991,   220]),
  'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 1])},
 {'input_ids': [tensor([ 5562,   705,    82,  1290,  1165, 15444,   284, 17004,   884, 31194,
            3513,   220]),
   tensor([26567,  2536,   689,   326,   262,  3437,   286,   884,   289, 31777,
            2512, 30181,   355, 29408,  1830,   460,   991,  1210,   503,   257,
            1402,   837,  2614,  2646,   351,   281,  7016,  3355,   404,   764,
             220]),
   tensor([   64, 19095, 17280,    12,  1941,    12,   727,   705,    82, 26781,
           19518,   220]),
   tensor([  533,   517,  7744,  1807,   832,   621,   287,   749,  4600,   826,
              12, 28973,   705,  7328,   220]),
   tensor([ 2188,   274,   284, 12986, 20428,   220])],
  'attention_mask': [tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]),
   tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
           1, 1, 1, 1, 1, 1

In [ ]:
# check what the pad token is set to (should be empty)
print(tokenizer.pad_token)

<PAD>


In [ ]:
# check what the eos token is set to
print(tokenizer.eos_token)

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorForLanguageModeling